# Tiny M1 LoRA learning diagnostic

This notebook performs the first training experiment: 12 fixed fictional units (4 F + 8 R), 84 training questions, 84 development questions and 120 optimizer attempts. It saves a diagnostic adapter, evaluates direct/multi-hop learning and checks fresh-base reload. This is not the canonical full M1. Attach HF_TOKEN in Kaggle Secrets to preserve artifacts privately before stopping the runtime.

In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect this notebook to a Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/training/m1_tiny.json').is_file(), 'Push the tiny training files first.'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__); assert torch.cuda.is_available()'], check=True)


In [ ]:
# The token is needed now because this experiment creates an adapter to preserve.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
api = HfApi(token=hf_token)
info = api.repo_info(artifact_repo, repo_type='model')
assert info.private, 'Use the configured private artifact repository.'
print('Private artifact storage is accessible. Token is not displayed.')


In [ ]:
import json

prep_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-prepare-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.training.tiny_m1', '--config', 'configs/training/m1_tiny.json', '--output-dir', str(prep_dir), '--mode', 'prepare'], check=True)
report = json.loads((prep_dir / 'masking_report.json').read_text())
assert report['rows'] == 84 and report['prefixes_verified'] and report['truncated_examples'] == 0
print('84 training examples prepared; answer masking verified.')


In [ ]:
# This cell runs actual LoRA training on the Kaggle GPU.
output_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-run-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.training.tiny_m1', '--config', 'configs/training/m1_tiny.json', '--output-dir', str(output_dir), '--mode', 'train'], check=True)


In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
print(json.dumps({'execution_complete': summary['execution_complete'], 'pilot': summary['pilot'], 'reload_predictions_identical': summary['reload_predictions_identical'], 'optimizer_updates': summary['optimizer_updates'], 'seconds': summary['seconds']}, indent=2))
print('A failed pilot is diagnostic evidence. Preserve it rather than relabeling it as full M1.')


In [ ]:
# Upload the adapter, optimizer snapshot, tokenizer settings and evidence privately.
import hashlib
from src.m0_smoke_test import write_json

remote_path = 'diagnostics/m1-tiny/' + output_dir.name
commit = api.upload_folder(repo_id=artifact_repo, repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve tiny M1 diagnostic')
restore_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-artifact-restore-', dir=remote_root))
for filename, expected in summary['artifact_files_sha256'].items():
    restored = hf_hub_download(repo_id=artifact_repo, filename=remote_path + '/' + filename, revision=commit.oid, token=hf_token, local_dir=restore_dir)
    assert hashlib.sha256(Path(restored).read_bytes()).hexdigest() == expected, filename
receipt = {'repo_id': artifact_repo, 'artifact_revision': commit.oid, 'path': remote_path, 'adapter_file_hashes_verified': True, 'canonical_m1': False}
write_json(output_dir / 'persistence.json', receipt)
api.upload_file(repo_id=artifact_repo, repo_type='model', path_or_fileobj=str(output_dir / 'persistence.json'), path_in_repo=remote_path + '/persistence.json', commit_message='Record verified adapter round-trip')
print(json.dumps(receipt, indent=2))


In [ ]:
# Export small evidence for Git; adapter and optimizer weights remain on private HF.
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
print('Stop the runtime only after the private receipt and small evidence ZIP are preserved.')
